In [13]:
from ortools.sat.python import cp_model
import random

class box:
    def __init__(self, n, celulas=None):
        self.n = n
        self.tamanho = n * n
        self.celulas = {}
        for (lin, col), val in (celulas or {}).items():
            self.add(lin, col, val)
    
    def add(self, lin, col, val=None):
        if not(1 <= lin <= self.tamanho and 1 <= col <= self.tamanho):
            raise ValueError("Célula fora da grelha")
        if val is not None and not (1 <= val <= self.tamanho):
            raise ValueError(f"Valor fora do intervalo [1, {self.tamanho}]")
        self.celulas[(lin,col)] = val
        return self

    def matriz(self):
        m = [[0] * self.tamanho for _ in range(self.tamanho)]
        for (lin, col), val in self.celulas.items():
            if val is not None:
                m[lin - 1][col - 1] = val
        return m

class cube(box):
    def __init__(self, n, ql, qc):
        super().__init__(n)
        if not(1 <= ql <= n and 1 <= qc <= n):
            raise ValueError("Índices errados!")
        for lin in range((ql - 1) * n + 1, ql * n + 1):
            for col in range((qc - 1) * n + 1, qc * n + 1):
                self.add(lin, col)

class path(box):
    def __init__(self, n, inicio, fim):
        super().__init__(n)
        (lin1, col1), (lin2, col2) = inicio, fim
        if lin1 != lin2 and col1 != col2:
            raise ValueError("Apenas devemos verificar linhas e/ou colunas!")
        if lin1 == lin2:
            step = 1 if col2 >= col1 else -1
            coords = [(lin1, c) for c in range(col1, col2 + step, step)]
        else:
            step = 1 if lin2 >= lin1 else -1
            coords = [(l, col1) for l in range(lin1, lin2 + step, step)]
        for lin, col in coords:
            self.add(lin, col)

def pistas_aleatorias(n, k=None, seed=None):
    tamanho = n * n
    if k is None:
        k = n
    if not (0 <= k <= tamanho * tamanho):
        raise ValueError(f"K tem que estar entre 0 e {tamanho * tamanho}")
    ran = random.Random(seed) 
    pistas = box(n)
    for c in ran.sample(range(tamanho * tamanho), k):  
        lin, col = c // tamanho + 1, c % tamanho + 1  
        pistas.add(lin, col, ran.randint(1, tamanho))
    return pistas

class Sudoku_Solver():
    def __init__(self, n):
        self.n = n
        self.tamanho = n * n
        self.modelo = cp_model.CpModel()
        self.x = {
            (lin, col): self.modelo.new_int_var(1, self.tamanho, f"x[{lin},{col}]")
            for lin in range(1, self.tamanho + 1)
            for col in range(1, self.tamanho + 1)
            }
        self.estado = None

    def add(self, *grupos):
        for g in grupos:
            variaveis = [self.x[cel] for cel in g.celulas]
            if len(variaveis) > 1:
                self.modelo.add_all_different(variaveis)
            for cel, val in g.celulas.items():
                if val is not None:
                    self.modelo.add(self.x[cel] == val)
        return self
    def resolver(self, tempo_max=10.0):
    
        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = tempo_max
        status = solver.solve(self.modelo)
        self.estado = solver.status_name(status)

        if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            return None

        return [[solver.value(self.x[(lin, col)]) for col in range(1, self.tamanho + 1)]
            for lin in range(1, self.tamanho + 1)]

def resolver_sudoku(n, k=None, seed=None):
    tamanho = n * n
    linhas = [path(n, (l, 1), (l, tamanho)) for l in range(1, tamanho + 1)]
    colunas = [path(n, (1, c), (tamanho, c)) for c in range(1, tamanho + 1)]
    blocos = [cube(n, ql, qc) for ql in range(1, n + 1) for qc in range(1, n + 1)]
    pistas = pistas_aleatorias(n, k, seed)
    return Sudoku_Solver(n).add(*linhas, *colunas, *blocos, pistas).resolver()





[[4, 2, 5, 3, 1, 6, 7, 9, 8],
 [8, 3, 6, 2, 7, 9, 1, 4, 5],
 [1, 7, 9, 5, 4, 8, 3, 2, 6],
 [5, 6, 4, 1, 2, 3, 8, 7, 9],
 [3, 1, 2, 8, 9, 7, 5, 6, 4],
 [9, 8, 7, 6, 5, 4, 2, 1, 3],
 [6, 5, 1, 4, 3, 2, 9, 8, 7],
 [7, 4, 3, 9, 8, 1, 6, 5, 2],
 [2, 9, 8, 7, 6, 5, 4, 3, 1]]